# 03. Distributed Data Processing with PySpark

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand PySpark for distributed data processing
- Implement PySpark to perform distributed data processing on large datasets
- Integrate PySpark with existing Python workflows
- Compare PySpark with Dask for distributed computing
- Apply PySpark to real-world large dataset scenarios

## 🔗 Where this fits

**Builds on:** Course 05 — Unit 5, lessons 01-02 (MapReduce, and Dask's partitioned model) — Spark is the JVM-era answer to the same problem, and the comparison is the point.

**Used later in:** Course 05 — Unit 5, lesson 07, which picks a tool per dataset size rather than by habit.

---

This notebook covers practical activities from **Course 05, Unit 5**:
- Data Processing using PySpark: Implementing PySpark to perform distributed data processing on large datasets and integrating with existing Python workflows

---


## The Story

**BEFORE**: You know Dask for distributed computing but don't know Spark for big data processing.

**AFTER**: You'll learn PySpark - industry-standard framework for distributed big data processing and analytics!

**Why this matters**: Distributed Data Processing with PySpark is essential for building complete, professional data science solutions!

---

# Unit 5 - Example 03: Distributed Data Processing with PySpark

## 🔗 Building on Example 02

**From Example 02 (Dask):**
- We learned Dask for distributed computing in Python
- Dask works well for Python-native workflows
- But for enterprise-scale distributed processing, we need Apache Spark

**This notebook introduces:**
- **PySpark** - Python API for Apache Spark
- **Distributed data processing** on large datasets
- **Integration** with existing Python workflows
- **Enterprise-scale** distributed computing

**This complements Dask with enterprise distributed computing!**

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Real data:** `montgomery_911_calls.csv` — 663,522 real 911 dispatches in
  Montgomery County, Pennsylvania (Dec 2015 – Jul 2020): location, timestamp,
  township and call type. The same table feeds Spark and the pandas fallback,
  so the comparison stays apples-to-apples.
- PySpark when installed; a clearly-labelled pandas fallback when it is not

**Outputs:** What you'll see when you run the cells

- Distributed filter / aggregate / transform on real emergency-call records
- Honest timings: no Spark numbers are printed if Spark did not run

---


In [1]:
# Try importing PySpark (requires Spark installation)
# PySpark may not be available on all systems - we'll provide fallback
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    PYSPARK_AVAILABLE = True
    print("✅ PySpark imported successfully!")
except ImportError:
    PYSPARK_AVAILABLE = False
    print("⚠️  PySpark not available. Install Spark for distributed processing:")
    print("   Note: Requires Apache Spark installation")
    print("   Continuing with the pandas fallback - real pandas, clearly labelled, never faked Spark.")

# Always import pandas/numpy for fallback
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

print("✅ Libraries imported!")

⚠️  PySpark not available. Install Spark for distributed processing:
   Note: Requires Apache Spark installation
   Continuing with the pandas fallback - real pandas, clearly labelled, never faked Spark.


✅ Libraries imported!


## Part 1: Introduction to PySpark

**PySpark** is the Python API for Apache Spark, a unified analytics engine for large-scale data processing.

**Key Features:**
- Distributed data processing across clusters
- In-memory computing for faster processing
- Integration with Hadoop ecosystem
- Support for SQL, streaming, and machine learning

**When to use PySpark vs Dask:**
- **PySpark**: Enterprise clusters, Hadoop integration, SQL queries, streaming
- **Dask**: Python-native workflows, smaller clusters, NumPy/Pandas compatibility

In [2]:
# WHAT: Print the banner and whether PySpark is truly available.
# WHY: The notebook runs real Spark when installed and a clearly-labeled pandas fallback when not - never fake distributed results.

print("=" * 70)
print("Example 03: Distributed Data Processing with PySpark")
print("=" * 70)
print("\n📚 Prerequisites: Example 02 (Dask) completed")
print("🔗 This notebook covers PySpark for distributed data processing")
print("🎯 Goal: Master PySpark for enterprise-scale distributed computing\n")

if PYSPARK_AVAILABLE:
    print("✅ PySpark is available - Using real distributed processing")
else:
    print("⚠️  PySpark not available - using the pandas fallback (real pandas, not simulated Spark)")
    print("   (Install Spark to use actual distributed processing)")

Example 03: Distributed Data Processing with PySpark

📚 Prerequisites: Example 02 (Dask) completed
🔗 This notebook covers PySpark for distributed data processing
🎯 Goal: Master PySpark for enterprise-scale distributed computing

⚠️  PySpark not available - using the pandas fallback (real pandas, not simulated Spark)
   (Install Spark to use actual distributed processing)


## Part 2: Creating Spark Session

**SparkSession** is the entry point for PySpark applications.

**Why SparkSession?**
- Manages Spark context and configuration
- Provides unified API for Spark SQL, DataFrames, and Datasets
- Handles distributed execution across cluster

In [3]:
# WHAT: Start a local SparkSession using all cores (master='local[*]').
# WHY: The SparkSession is the entry point to Spark - the same code scales from a laptop to a cluster by changing the master URL.

if PYSPARK_AVAILABLE:
    # Create SparkSession for distributed processing
    spark = (SparkSession.builder
             .appName("Course05_PySpark_Example")
             .master("local[*]")  # Use all available cores locally
             .getOrCreate())
    
    print("✅ SparkSession created successfully!")
    print(f"Spark version: {spark.version}")
    print(f"Spark master: {spark.sparkContext.master}")
else:
    print("⚠️  PySpark unavailable - using pandas for the demonstration below")
    print("   (Same operations, but single-machine processing)")

⚠️  PySpark unavailable - using pandas for the demonstration below
   (Same operations, but single-machine processing)


## Part 3: Loading the Real Dataset

We load a real, reasonably large table — every 911 call dispatched in
Montgomery County, PA between December 2015 and July 2020 — to demonstrate
PySpark's distributed processing on records that were never generated by us.


In [4]:
# WHAT: Load the real Montgomery County 911 dispatch log.
# WHY: One shared real dataset feeds either engine, keeping the comparison apples-to-apples.

DATA_DIR = '../../../Course 04/datasets/raw/'

sample_data = pd.read_csv(DATA_DIR + 'montgomery_911_calls.csv',
                          usecols=['lat', 'lng', 'title', 'timeStamp', 'twp'],
                          parse_dates=['timeStamp'])

# 'title' looks like 'EMS: BACK PAINS/INJURY' - the part before ':' is the service.
sample_data['service'] = sample_data['title'].str.split(':').str[0]
sample_data['hour'] = sample_data['timeStamp'].dt.hour
sample_data = sample_data[['lat', 'lng', 'service', 'hour', 'twp']].copy()
sample_data.insert(0, 'id', range(len(sample_data)))
n_rows = len(sample_data)

print(f"✅ Real data loaded: {n_rows:,} 911 calls")
print(f"Data shape: {sample_data.shape}")
print(f"Services: {sample_data['service'].value_counts().to_dict()}")
print(f"Missing township values (real gaps): {sample_data['twp'].isna().sum()}")
print("\nFirst few rows:")
print(sample_data.head())

✅ Real data loaded: 663,522 911 calls
Data shape: (663522, 6)
Services: {'EMS': 332692, 'Traffic': 230208, 'Fire': 100622}
Missing township values (real gaps): 293

First few rows:
   id        lat        lng service  hour                twp
0   0  40.297876 -75.581294     EMS    17        NEW HANOVER
1   1  40.258061 -75.264680     EMS    17  HATFIELD TOWNSHIP
2   2  40.121182 -75.351975    Fire    14         NORRISTOWN
3   3  40.116153 -75.343513     EMS    16         NORRISTOWN
4   4  40.251492 -75.603350     EMS    16   LOWER POTTSGROVE


## Part 4: Loading Data into PySpark

PySpark DataFrames are distributed collections of data organized into named columns.

In [5]:
# WHAT: Write the real data to CSV and load it as a Spark DataFrame (or keep pandas in fallback mode).
# WHY: inferSchema and partition counts show how Spark plans distributed reads.

if PYSPARK_AVAILABLE:
    # Spark reads from files, so write the prepared table out once.
    # (This CSV is a scratch artefact derived from the real dataset; it is gitignored.)
    csv_path = 'montgomery_911_prepared.csv'
    sample_data.to_csv(csv_path, index=False)

    # Load into PySpark DataFrame
    df_spark = spark.read.csv(csv_path, header=True, inferSchema=True)

    print("✅ Data loaded into PySpark DataFrame!")
    print(f"Number of partitions: {df_spark.rdd.getNumPartitions()}")
    print(f"Total rows: {df_spark.count():,}")
    print("\nSchema:")
    df_spark.printSchema()
    print("\nFirst few rows:")
    df_spark.show(5)
    df_pandas = sample_data.copy()
else:
    print("⚠️  PySpark not installed - using the pandas fallback on the SAME real data")
    df_pandas = sample_data.copy()
    print(f"Data shape: {df_pandas.shape}")
    print("\nFirst few rows:")
    print(df_pandas.head())

⚠️  PySpark not installed - using the pandas fallback on the SAME real data
Data shape: (663522, 6)

First few rows:
   id        lat        lng service  hour                twp
0   0  40.297876 -75.581294     EMS    17        NEW HANOVER
1   1  40.258061 -75.264680     EMS    17  HATFIELD TOWNSHIP
2   2  40.121182 -75.351975    Fire    14         NORRISTOWN
3   3  40.116153 -75.343513     EMS    16         NORRISTOWN
4   4  40.251492 -75.603350     EMS    16   LOWER POTTSGROVE


## Part 5: Distributed Data Processing Operations

PySpark performs operations in a distributed manner across partitions.

In [6]:
# WHAT: Run filtering, aggregation, and column transformations through Spark (or pandas).
# WHY: These are the same operations from Unit 2 - only the ENGINE changed, which is the point of DataFrame APIs.

print("\n" + "=" * 70)
print("PART 5: Distributed Data Processing Operations")
print("=" * 70)

if PYSPARK_AVAILABLE:
    # Filtering (distributed)
    print("\n1. Filtering data (distributed across partitions):")
    filtered = df_spark.filter(df_spark['hour'] >= 18)
    print(f"   Calls dispatched from 18:00 onwards: {filtered.count():,}")

    # Aggregations (distributed)
    print("\n2. Aggregations (distributed):")
    aggregated = df_spark.groupBy('service').agg(
        F.avg('hour').alias('avg_hour'),
        F.count('*').alias('count')
    )
    print("   Grouped by emergency service:")
    aggregated.show()

    # Transformations (distributed)
    print("\n3. Transformations (distributed):")
    transformed = df_spark.withColumn('is_night',
                                      (df_spark['hour'] < 6) | (df_spark['hour'] >= 22))
    print("   Added new column 'is_night'")
    transformed.select('id', 'service', 'hour', 'is_night').show(5)
else:
    # Pandas fallback on the same real data
    print("\n1. Filtering data:")
    filtered = df_pandas[df_pandas['hour'] >= 18]
    print(f"   Calls dispatched from 18:00 onwards: {len(filtered):,} "
          f"({len(filtered) / len(df_pandas):.1%} of all calls)")

    print("\n2. Aggregations:")
    aggregated = df_pandas.groupby('service').agg(
        avg_hour=('hour', 'mean'),
        count=('id', 'count'))
    print("   Grouped by emergency service:")
    print(aggregated.round(2))

    print("\n3. Transformations:")
    df_pandas['is_night'] = (df_pandas['hour'] < 6) | (df_pandas['hour'] >= 22)
    print("   Added new column 'is_night'")
    print(df_pandas[['id', 'service', 'hour', 'is_night']].head())
    print(f"   Night calls (22:00-06:00): {df_pandas['is_night'].sum():,} "
          f"({df_pandas['is_night'].mean():.1%})")


PART 5: Distributed Data Processing Operations

1. Filtering data:
   Calls dispatched from 18:00 onwards: 159,707 (24.1% of all calls)

2. Aggregations:
   Grouped by emergency service:
         avg_hour   count
service                  
EMS         12.74  332692
Fire        13.33  100622
Traffic     13.35  230208

3. Transformations:
   Added new column 'is_night'
   id service  hour  is_night
0   0     EMS    17     False
1   1     EMS    17     False
2   2    Fire    14     False
3   3     EMS    16     False
4   4     EMS    16     False
   Night calls (22:00-06:00): 103,901 (15.7%)


## Part 6: Performance Comparison

Comparing PySpark (distributed) vs Pandas (single-machine) performance.

In [7]:
# WHAT: Time a filter+groupby pipeline on the available engine(s).
# WHY: Spark's overhead loses on a few hundred thousand local rows - its win is horizontal
#      scale, and honest timing shows both sides.

print("\n" + "=" * 70)
print("PART 6: Performance Comparison")
print("=" * 70)

# Test complex operation: filtering + aggregation
if PYSPARK_AVAILABLE:
    print("\n⏱️  Testing PySpark (distributed) performance...")
    start_time = time.time()

    result_spark = (df_spark
                    .filter(df_spark['hour'] >= 18)
                    .groupBy('service')
                    .agg(F.avg('hour').alias('avg_hour'))
                    .collect())

    spark_time = time.time() - start_time
    print(f"   ✅ PySpark (distributed): {spark_time:.4f} seconds")
    print(f"   Results: {len(result_spark)} groups")
else:
    print("\n⏱️  PySpark not installed on this machine - no Spark timing measured.")
    print("   (PySpark would distribute this work across multiple cores/machines.)")
    spark_time = None

# Pandas (single-machine)
print("\n⏱️  Testing Pandas (single-machine) performance...")
start_time = time.time()

result_pandas = (df_pandas[df_pandas['hour'] >= 18]
                 .groupby('service')['hour']
                 .mean())

pandas_time = time.time() - start_time
print(f"   ✅ Pandas (single-machine): {pandas_time:.4f} seconds "
      f"on {len(df_pandas):,} real rows")
print(result_pandas.round(2).to_string())

# Comparison
if PYSPARK_AVAILABLE:
    speedup = pandas_time / spark_time
    print(f"\n📊 Speedup: {speedup:.2f}x with PySpark (measured on this machine)")
    print("   (Spark's advantage grows with cluster size and data volume)")
else:
    print(f"\n📊 No comparison possible: only the pandas baseline ({pandas_time:.4f}s)")
    print(f"   was measured here on {len(df_pandas):,} rows. Distributed speedups require")
    print("   an actual Spark installation - install PySpark and re-run to measure them.")


PART 6: Performance Comparison

⏱️  PySpark not installed on this machine - no Spark timing measured.
   (PySpark would distribute this work across multiple cores/machines.)

⏱️  Testing Pandas (single-machine) performance...
   ✅ Pandas (single-machine): 0.0103 seconds on 663,522 real rows
service
EMS        20.19
Fire       20.02
Traffic    19.85

📊 No comparison possible: only the pandas baseline (0.0103s)
   was measured here on 663,522 rows. Distributed speedups require
   an actual Spark installation - install PySpark and re-run to measure them.


## Part 7: Integration with Python Workflows

PySpark integrates seamlessly with existing Python workflows.

In [8]:
# WHAT: Convert a Spark result to pandas and hand it to scikit-learn.
# WHY: Real pipelines mix engines - Spark crunches the big data, then pandas/sklearn handle the reduced result.

print("\n" + "=" * 70)
print("PART 7: Integration with Python Workflows")
print("=" * 70)

from sklearn.preprocessing import StandardScaler

if PYSPARK_AVAILABLE:
    # Convert PySpark DataFrame to Pandas (for integration)
    print("\n1. Converting PySpark DataFrame to Pandas:")
    df_pandas_from_spark = df_spark.limit(1000).toPandas()
    print(f"   Converted {len(df_pandas_from_spark):,} rows to Pandas")
    print("   Now you can use pandas/numpy/scikit-learn on this data")

    print("\n2. Using with existing Python libraries:")
    scaled_data = scaler_input = df_pandas_from_spark[['lat', 'lng', 'hour']].dropna()
    scaled_data = StandardScaler().fit_transform(scaler_input)
    print(f"   Scaled data shape: {scaled_data.shape}")
    print("   ✅ PySpark integrates with scikit-learn, pandas, numpy!")
else:
    print("\n⚠️  PySpark not installed - showing the same integration in pandas:")
    subset = df_pandas[['lat', 'lng', 'hour']].dropna().head(1000)
    scaled_data = StandardScaler().fit_transform(subset)
    print(f"   Scaled data shape: {scaled_data.shape}")
    print(f"   Column means after scaling: {scaled_data.mean(axis=0).round(6)}")
    print("\n   In real PySpark, you would:")
    print("   1. Process the full dataset distributed across a cluster")
    print("   2. Convert the (much smaller) RESULT to pandas - as above")
    print("   3. Use scikit-learn, pandas, numpy on that result")
    print("   4. Write results back to distributed storage")


PART 7: Integration with Python Workflows



⚠️  PySpark not installed - showing the same integration in pandas:
   Scaled data shape: (1000, 3)
   Column means after scaling: [0. 0. 0.]

   In real PySpark, you would:
   1. Process the full dataset distributed across a cluster
   2. Convert the (much smaller) RESULT to pandas - as above
   3. Use scikit-learn, pandas, numpy on that result
   4. Write results back to distributed storage


## Part 8: Summary

**Key Takeaways:**

1. **PySpark** provides distributed data processing for large datasets
2. **Distributed operations** run across multiple cores/machines
3. **Integration** with Python workflows (pandas, scikit-learn, numpy)
4. **Performance** scales with cluster size

**When to use PySpark:**
- Very large datasets (billions of rows)
- Enterprise clusters and Hadoop integration
- SQL queries on distributed data
- Streaming data processing

**When to use Dask (from Example 02):**
- Python-native workflows
- Smaller clusters
- NumPy/Pandas compatibility
- Interactive data science

In [9]:
# WHAT: Print the summary and stop the SparkSession.
# WHY: Stopping the session releases cluster resources - good hygiene in shared environments.

print("\n" + "=" * 70)
print("SUMMARY: PySpark for Distributed Data Processing")
print("=" * 70)

print("\n✅ What you learned:")
print("   1. PySpark for distributed data processing")
print("   2. Creating SparkSession and DataFrames")
print("   3. Distributed operations (filtering, aggregation, transformation)")
print("   4. Integration with Python workflows")
if PYSPARK_AVAILABLE:
    print("   5. Performance comparison: PySpark vs pandas (measured above)")
else:
    print("   5. How Spark WOULD distribute work (PySpark did not run here -")
    print("      the demos above used the clearly-labeled pandas fallback)")

print("\n🔗 Next steps:")
print("   - Example 04: RAPIDS workflows (GPU acceleration)")
print("   - Example 05: Production pipelines")
print("   - Example 06: Performance optimization")

if PYSPARK_AVAILABLE:
    spark.stop()
    print("\n✅ SparkSession stopped")
else:
    print("\n💡 To use PySpark:")
    print("   1. Install Apache Spark: https://spark.apache.org/downloads.html")
    print("   2. Install PySpark: pip install pyspark")
    print("   3. Run this notebook again for distributed processing")


SUMMARY: PySpark for Distributed Data Processing

✅ What you learned:
   1. PySpark for distributed data processing
   2. Creating SparkSession and DataFrames
   3. Distributed operations (filtering, aggregation, transformation)
   4. Integration with Python workflows
   5. How Spark WOULD distribute work (PySpark did not run here -
      the demos above used the clearly-labeled pandas fallback)

🔗 Next steps:
   - Example 04: RAPIDS workflows (GPU acceleration)
   - Example 05: Production pipelines
   - Example 06: Performance optimization

💡 To use PySpark:
   1. Install Apache Spark: https://spark.apache.org/downloads.html
   2. Install PySpark: pip install pyspark
   3. Run this notebook again for distributed processing


## 📚 References

1. Zaharia, M., Chowdhury, M., Das, T., et al. (2012). *Resilient Distributed Datasets: A Fault-Tolerant Abstraction for In-Memory Cluster Computing*. NSDI 2012. <https://www.usenix.org/conference/nsdi12/technical-sessions/presentation/zaharia>
2. Zaharia, M., Xin, R. S., Wendell, P., et al. (2016). *Apache Spark: A Unified Engine for Big Data Processing*. Communications of the ACM, 59(11), 56-65. <https://doi.org/10.1145/2934664>
3. Dean, J., & Ghemawat, S. (2004). *MapReduce: Simplified Data Processing on Large Clusters*. OSDI 2004. <https://research.google/pubs/mapreduce-simplified-data-processing-on-large-clusters/>